In [1]:
import os
from dotenv import load_dotenv

load_dotenv(
    r"C:\Users\guruz\OneDrive\Desktop\Lanchan\.env",
    override=True
)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("API key not found!")

print("API key loaded successfully")

API key loaded successfully


In [2]:
from langchain_openai import ChatOpenAI

llm_openai = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0
)

In [6]:
## Load Doument
from langchain_community.document_loaders import PyPDFLoader

pdf_files = [
    "./doc/01_Traffic_Rules_and_Road_Safety.pdf",
    "./doc/02_Driving_Licence_Process.pdf",
    "./doc/03_School_Policy.pdf",
    "./doc/04_Driving_School_SOP.pdf",
    "./doc/05_FAQ.pdf"
]

docs = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(pdf_file)
    documents = loader.load()
    for doc in documents:
        doc.metadata={
            "document_name": pdf_file.split("/")[-1].replace(".pdf", ""),
            "source": pdf_file,
            "page": doc.metadata["page"],
            "category": "driving_school",
            "document_type": "knowledge_base"
        }
    docs.extend(documents)

    print(f"Loaded: {pdf_file} | Pages: {len(documents)}")

print("\nTotal pages loaded:", len(docs))

Loaded: ./doc/01_Traffic_Rules_and_Road_Safety.pdf | Pages: 5
Loaded: ./doc/02_Driving_Licence_Process.pdf | Pages: 5
Loaded: ./doc/03_School_Policy.pdf | Pages: 6
Loaded: ./doc/04_Driving_School_SOP.pdf | Pages: 7
Loaded: ./doc/05_FAQ.pdf | Pages: 4

Total pages loaded: 27


Convert in to chunks

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

spliter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
chunk = spliter.split_documents(docs)


In [9]:
print(type(docs))
print(type(docs[0]))
print(docs[0].metadata)

<class 'list'>
<class 'langchain_core.documents.base.Document'>
{'document_name': '01_Traffic_Rules_and_Road_Safety', 'source': './doc/01_Traffic_Rules_and_Road_Safety.pdf', 'page': 0, 'category': 'driving_school', 'document_type': 'knowledge_base'}


## Create a Embedding Model


In [10]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [15]:
from langchain_community.vectorstores import Chroma
vector_db = Chroma.from_documents(
    embedding=embedding_model,
    documents=chunk,
    persist_directory="./vector/"
)


In [17]:
context = vector_db.similarity_search("til me about driving school")

In [18]:
llm_openai.invoke(f"till me about driving school using this context: {context}").content

'Here’s a concise summary of the driving school information from the documents you provided.\n\nQuick overview\n- The school has been operating since about 2012–13 and runs under the legal framework of the Motor Vehicles Act and the Central Motor Vehicles Rules.\n- It provides classroom/study material and practical driving training, plus app-based lectures and practice tests (free and paid batches).\n\nLegal / licensing\n- A driving school must hold a licence in Form 11 (Central Motor Vehicles Rules). The original licence is displayed in the school office; you may ask to see it.\n- Licence validity and renewal follow the rules (Form 11 valid 5 years; renewal in Form 13).\n- The school issues a driving school certificate (Form 5) to every student who completes the course (to be attached to your licence application).\n\nServices offered\n- Assistance with learner licence application and test preparation.\n- Practical driving training for the vehicle classes you enroll for.\n- Help with p